# 1. scGBM marker-based step2 粗注释

读取 `0_integrate_two_datasets.ipynb` 输出的 `adata1.h5ad`，按照 seekgene、snGBM、三位置 GBM 共用的 marker gene 方案进行粗注释。

保存逻辑：输入来自 `write/0_integrate_two_datasets/adata1.h5ad`，本步骤输出写入 `write/1_preprocess_scRNAdata/adata2.h5ad`。

### 1. 包和路径

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
sns.set_theme(style="white", font="Arial", rc={"figure.dpi": 120})

BASE_DIR = Path(".").resolve()
INTEGRATE_WRITE_DIR = BASE_DIR / "write" / "0_integrate_two_datasets"
WRITE_DIR = BASE_DIR / "write" / "1_preprocess_scRNAdata"
FIG_DIR = BASE_DIR / "figures" / "1_preprocess_scRNAdata"
H5AD_BEFORE_MARKER = INTEGRATE_WRITE_DIR / "adata1.h5ad"
H5AD_AFTER_MARKER = WRITE_DIR / "adata2.h5ad"
FIG_DIR.mkdir(parents=True, exist_ok=True)
WRITE_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(FIG_DIR)

print("BASE_DIR:", BASE_DIR)
print("H5AD_BEFORE_MARKER:", H5AD_BEFORE_MARKER)
print("H5AD_AFTER_MARKER:", H5AD_AFTER_MARKER)
print("WRITE_DIR:", WRITE_DIR)
print("FIG_DIR:", FIG_DIR)


### 2. 读取数据

In [ ]:
adata = sc.read_h5ad(H5AD_BEFORE_MARKER)
adata.var_names_make_unique()
if "log1p" in adata.uns:
    adata.uns["log1p"]["base"] = None

print(adata)
print("obs columns:", list(adata.obs.columns))
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))

assert "leiden" in adata.obs.columns, "缺少 leiden cluster"
assert "X_umap" in adata.obsm, "缺少 X_umap"
assert "counts" in adata.layers, "缺少 counts layer"
assert adata.raw is None, "当前 scGBM step2 输入不应依赖 raw，本轮统一使用 adata.X 打分"
print("读取完成")

### 3. 与其他数据集一致的 marker gene list

In [ ]:
# 与 seekgene、snGBM、三位置 GBM 保持一致
marker_gene_sets = {
    "Oligodendrocyte": ["MAG", "CLDN11", "APLP1", "TMEM144", "CNDP1", "EDIL3", "LARP6", "MOG", "AMER2", "TUBB4A"],
    "MES-like": ["VIM", "CD44", "CHI3L1", "HILPDA", "DDIT3", "ENO2"],
    "AC-like": ["GFAP", "S100B", "HOPX", "SLC1A3", "MLC1"],
    "OPC-like": ["PLP1", "OLIG1", "OMG", "TNR", "ALCAM"],
    "NPC-like": ["SOX4", "SOX11", "DCX", "CD24", "STMN1", "STMN2"],
    "NK/T": ["CD3D", "CD3E", "TRBC2", "CD3G", "CD2", "TRAC", "IL7R", "TRBC1", "LCK", "SKAP1", "CD48"],
    "Prol.NK/T": ["MKI67", "TOP2A", "CD3D", "CD3E", "TRBC2", "CD3G"],
    "Pericyte": ["COL1A2", "COL6A2", "PDGFRB", "COL3A1", "EDNRA", "LUM", "COL1A1", "PLAC9", "FRZB"],
    "Endothelial": ["ESAM", "FLT1", "RAMP2", "VWF", "EGFL7", "SLC9A3R2", "CAVIN2", "ADGRL4", "ABLIM1", "ABCB1"],
    "Monocyte": ["FCN1", "VCAN", "S100A8", "S100A9"],
    "Microglia": ["TMEM119", "CX3CR1", "TREM2", "GPNMB", "CTSD", "CD68"],
    "Macrophage": ["SPP1", "APOE", "GPNMB", "CTSD", "CD68", "C1QA", "C1QB"],
    "CDC": ["FCER1A", "HLA-DPB1", "CD83", "AREG"],
    "Neutrophil": ["S100A8", "FCGR3B", "IFITM2"],
}

valid_marker_sets = {
    celltype: [gene for gene in genes if gene in adata.var_names]
    for celltype, genes in marker_gene_sets.items()
}
valid_marker_sets = {k: v for k, v in valid_marker_sets.items() if len(v) > 0}

presence = []
for celltype, genes in marker_gene_sets.items():
    valid = [gene for gene in genes if gene in adata.var_names]
    missing = [gene for gene in genes if gene not in adata.var_names]
    presence.append({
        "celltype": celltype,
        "n_total": len(genes),
        "n_valid": len(valid),
        "valid_genes": ";".join(valid),
        "missing_genes": ";".join(missing),
    })

presence_df = pd.DataFrame(presence)
presence_df.to_csv(WRITE_DIR / "scGBM_marker_gene_presence.csv", index=False)
display(presence_df)

### 3.1 当前粗注释 marker gene 展示

In [ ]:
marker_display_rows = []
for celltype, genes in marker_gene_sets.items():
    present_genes = [gene for gene in genes if gene in adata.var_names]
    missing_genes = [gene for gene in genes if gene not in adata.var_names]
    marker_display_rows.append({
        "celltype": celltype,
        "n_marker": len(genes),
        "n_present": len(present_genes),
        "marker_genes": ", ".join(genes),
        "present_genes": ", ".join(present_genes),
        "missing_genes": ", ".join(missing_genes),
    })

marker_display_df = pd.DataFrame(marker_display_rows)
marker_display_df.to_csv(WRITE_DIR / "scGBM_nb1_marker_gene_display.csv", index=False)
display(marker_display_df)

### 4. 按统一 marker 逻辑打分并注释

In [ ]:
for celltype, genes in valid_marker_sets.items():
    if len(genes) >= 2:
        sc.tl.score_genes(adata, gene_list=genes, score_name=f"score_{celltype}", use_raw=False)

score_cols = [f"score_{celltype}" for celltype, genes in valid_marker_sets.items() if len(genes) >= 2]
scores_df = adata.obs[score_cols].copy()

MIN_SCORE_DIFF = 0.05
sorted_scores = np.sort(scores_df.values, axis=1)
top1 = sorted_scores[:, -1]
top2 = sorted_scores[:, -2]
score_diff = top1 - top2

predicted = scores_df.idxmax(axis=1).str.replace("score_", "", regex=False)
low_confidence = score_diff < MIN_SCORE_DIFF
predicted[low_confidence] = "Ambiguous"
adata.obs["predicted_celltype"] = predicted.astype("category")
adata.obs["annotation_confidence"] = score_diff

cluster_col = "leiden"
cluster_records = []
cluster_anno = {}
for cl in sorted(adata.obs[cluster_col].astype(str).unique(), key=lambda x: int(x)):
    mask = adata.obs[cluster_col].astype(str).eq(cl)
    votes = adata.obs.loc[mask, "predicted_celltype"].astype(str)
    non_ambig = votes[votes != "Ambiguous"]
    if len(non_ambig) > 0:
        top = non_ambig.value_counts()
        label = top.index[0]
        support = top.iloc[0] / len(votes)
    else:
        label = "Unknown"
        support = 0
    cluster_anno[cl] = label
    cluster_records.append({
        "leiden": cl,
        "celltype": label,
        "support_fraction": support,
        "n_cells": int(mask.sum()),
        "n_ambiguous": int((votes == "Ambiguous").sum()),
    })

adata.obs["celltype"] = adata.obs[cluster_col].astype(str).map(cluster_anno).astype("category")
adata.obs["cell_type"] = adata.obs["celltype"].astype(str).astype("category")
adata.obs["self_annotation"] = adata.obs["cell_type"].astype(str).astype("category")

cluster_score = adata.obs.groupby(cluster_col, observed=True)[score_cols].mean()
cluster_score.to_csv(WRITE_DIR / "scGBM_cluster_marker_scores.csv")

helper = pd.DataFrame(cluster_records)
helper.to_csv(WRITE_DIR / "scGBM_cluster_annotation_helper.csv", index=False)
display(helper)
print(adata.obs["celltype"].value_counts())

### 5. 统一 marker dotplot

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=valid_marker_sets,
    groupby="celltype",
    use_raw=False,
    standard_scale="var",
    dendrogram=True,
    show=False,
)
plt.savefig(FIG_DIR / "merged_marker_dotplot.pdf", bbox_inches="tight")
plt.savefig(FIG_DIR / "merged_marker_dotplot.png", dpi=300, bbox_inches="tight")
plt.show()

sc.pl.dotplot(
    adata,
    var_names=valid_marker_sets,
    groupby="leiden",
    use_raw=False,
    figsize=(36, 12),
    var_group_rotation=90,
    standard_scale="var",
    dendrogram=False,
    show=False,
)
plt.savefig(FIG_DIR / "dotplot_markers_leiden.pdf", bbox_inches="tight")
plt.savefig(FIG_DIR / "dotplot_markers_leiden.png", dpi=300, bbox_inches="tight")
plt.show()

### 6. UMAP 展示

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 7))
sc.pl.umap(adata, color="leiden", ax=axes[0], show=False,
           title="Clusters", legend_loc="on data", legend_fontsize=6, frameon=False)
sc.pl.umap(adata, color="celltype", ax=axes[1], show=False,
           title="Cell Type", legend_fontsize=7, frameon=False)
sc.pl.umap(adata, color="sample", ax=axes[2], show=False,
           title="Sample", frameon=False)
sc.pl.umap(adata, color="tissue_type", ax=axes[3], show=False,
           title="Tissue", frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "merged_celltype_UMAP.pdf", bbox_inches="tight")
plt.savefig(FIG_DIR / "merged_celltype_UMAP.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.umap(
    adata,
    color="celltype",
    legend_loc="on data",
    legend_fontsize=7,
    frameon=False,
    size=4,
    add_outline=True,
    outline_width=(0.1, 0.05),
    ax=ax,
    show=False,
    title=f"scGBM Cell Types by unified markers (n={adata.n_obs:,})",
)
plt.tight_layout()
plt.savefig(FIG_DIR / "enhanced_umap_celltype.pdf", bbox_inches="tight")
plt.savefig(FIG_DIR / "enhanced_umap_celltype.png", dpi=300, bbox_inches="tight")
plt.show()

### 7. 保存 marker 后对象

In [ ]:
adata.write_h5ad(H5AD_AFTER_MARKER)
print("saved:", H5AD_AFTER_MARKER)
print(adata)

adata_test = sc.read_h5ad(H5AD_AFTER_MARKER, backed="r")
assert adata_test.n_obs == adata.n_obs
assert "cell_type" in adata_test.obs.columns
assert "celltype" in adata_test.obs.columns
assert "self_annotation" in adata_test.obs.columns
print("readback ok:", adata_test.shape)
adata_test.file.close()